# 中证800 V98 Money2 外部新信息准入与增量验证

本实验只研究 Money2 之外的真正新增信息，不重复 V90 的静态分析师代理。

- **P1 真正分析师预期修正**：必须有逐条研报发布日期、预测目标期、机构/分析师和预测值；默认只做数据准入，不用 jqfactor 静态代理冒充。
- **P2 股东行为与解禁**：已完成增减持、已公告的未来解禁。标准表不保证覆盖预减持和回购，缺口会明确输出。
- **P3 实际财务加速度与公告后反应**：使用 `get_fundamentals(..., date=feature_date)` 的时点数据；它不是“相对分析师一致预期的 surprise”。
- **P4 持有人结构**：股东户数、前十大流通股东集中度和可识别的机构持股占比。

实验以 Money2 为唯一 baseline，逐族比较，再评估全部可用信息族。所有查询均缓存，训练逐模型释放内存。

## 0. 导入、进度条与兼容层

In [ ]:
from jqdata import *
import builtins as _bi
import datetime as _dt
import gc
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=20):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 冻结配置与预注册门槛

In [ ]:
PROJECT_DIR = Path.cwd()
PRODUCT_VERSION = "v98"
RUN_NAME = "money2_external_information_gate"
RUN_TIMESTAMP = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = PROJECT_DIR / "csi800_ml_v98_external_information_outputs"
FIG_DIR = OUT_DIR / "figures"
QA_DIR = OUT_DIR / "qa"
CACHE_DIR = OUT_DIR / "source_cache"
for path in [OUT_DIR, FIG_DIR, QA_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PANEL_PATH_OVERRIDE = None
PANEL_CANDIDATES = [
    Path("csi800_ml_v96_money_quality_outputs/v96_money2_enriched_panel.csv"),
    Path("../csi800_ml_v96_money_quality_outputs/v96_money2_enriched_panel.csv"),
    Path("v96_money2_enriched_panel.csv"),
]

STOCK_COL = "stock"
DATE_COL = "rebalance_date"
NEXT_DATE_COL = "next_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "alpha_1m"

FORCE_REFETCH = False
SOURCE_BATCH_SIZE = 150
SOURCE_CACHE_VERSION = "v3_windowed_compact"
MAX_SAFE_CHUNK_ROWS = 4999
TABLE_FETCH_POLICIES = {
    "STK_SHAREHOLDERS_SHARE_CHANGE": {"stock_batch_size": 120, "window_years": 2},
    "STK_LIMITED_SHARES_LIST": {"stock_batch_size": 150, "window_years": 4},
    "STK_HOLDER_NUM": {"stock_batch_size": 120, "window_years": 2},
    "STK_SHAREHOLDER_FLOATING_TOP10": {"stock_batch_size": 60, "window_years": 1},
}
TABLE_REQUIRED_FIELDS = {
    "STK_SHAREHOLDERS_SHARE_CHANGE": ["code", "pub_date", "end_date", "type", "change_ratio"],
    "STK_LIMITED_SHARES_LIST": [
        "code", "pub_date", "expected_unlimited_date", "expected_unlimited_ratio",
        "actual_unlimited_date", "actual_unlimited_ratio",
    ],
    "STK_HOLDER_NUM": ["code", "pub_date", "end_date", "share_holders", "a_share_holders"],
    "STK_SHAREHOLDER_FLOATING_TOP10": [
        "code", "pub_date", "end_date", "shareholder_name", "shareholder_class", "share_ratio",
    ],
}
INSTITUTION_KEYWORDS = [
    "基金", "保险", "证券", "社保", "资产管理", "投资管理", "信托", "银行",
    "QFII", "RQFII", "FUND", "ASSET", "CAPITAL",
]
PRED_TOP_K = 8
TRUE_TOP_N = 20
BOARD_CAPS = {"chinext": 3, "star": 2}
FIXED_ITER = 120
NUM_THREADS = 4
CORR_THRESHOLD_V46 = 0.70
CORR_THRESHOLD_INCREMENTAL = 0.95
MIN_INCREMENTAL_TRAIN_COVERAGE = 0.05
MIN_TRAIN_MONTHS = 30
MIN_SOURCE_COVERAGE = 0.70
BASELINE_RANK_IC_ATOL = 1e-10
BASELINE_TOP8_EDGE_ATOL = 1e-7

BASE_PARAMS = {
    "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
    "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200,
    "feature_fraction": 1.0, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 0.3, "verbose": -1,
}
CORE_CUTOFFS = ["2021-12-31", "2022-12-31", "2023-12-31", "2024-12-31", "2025-12-31"]
CORE_SEEDS = [42, 2024, 2026]
FEATURE_FRACTION = 1.0
UPDATE_SEED = 42
UPDATE_PAIRS = [
    {"pair_id": "2021_to_2022_eval2023", "old_cutoff": "2021-12-31", "new_cutoff": "2022-12-31", "eval_start": "2023-01-01", "eval_end": "2023-12-31"},
    {"pair_id": "2022_to_2023_eval2024", "old_cutoff": "2022-12-31", "new_cutoff": "2023-12-31", "eval_start": "2024-01-01", "eval_end": "2024-12-31"},
    {"pair_id": "2023_to_2024_eval2025", "old_cutoff": "2023-12-31", "new_cutoff": "2024-12-31", "eval_start": "2025-01-01", "eval_end": "2025-12-31"},
    {"pair_id": "2024_to_2025_eval2026", "old_cutoff": "2024-12-31", "new_cutoff": "2025-12-31", "eval_start": "2026-01-01", "eval_end": "2026-12-31"},
]
RUN_UPDATE_STABILITY = True

BOOTSTRAP_SAMPLES = 2000
BOOTSTRAP_BLOCK_MONTHS = 3
BOOTSTRAP_SEED = 9801

# 单项新增信息族必须全部通过，才进入候选。
ACCEPT_MIN_DELTA_TOP8_EDGE = 0.002
ACCEPT_MIN_DELTA_RANK_IC = -0.002
ACCEPT_MIN_POSITIVE_YEARS = 3
ACCEPT_MIN_POSITIVE_SEEDS = 2
ACCEPT_MIN_BOOTSTRAP_PROBABILITY = 0.65
ACCEPT_MIN_UPDATE_SCORE_CORR = 0.65
ACCEPT_MIN_UPDATE_TOP8_OVERLAP = 0.25

# P1 必须由真实逐条分析师数据接入。留空时只输出“不可验证”，不使用代理。
ANALYST_SOURCE_NAME = None
ANALYST_REQUIRED_FIELDS = [
    "stock_code", "publication_date", "forecast_period", "institution_or_analyst",
    "forecast_eps_or_net_profit",
]

SMOKE_MODE = False
SMOKE_MAX_MONTHS = 2
if SMOKE_MODE:
    CORE_CUTOFFS = CORE_CUTOFFS[-1:]
    CORE_SEEDS = CORE_SEEDS[:1]
    UPDATE_PAIRS = UPDATE_PAIRS[-1:]
    BOOTSTRAP_SAMPLES = 100

print("OUT_DIR:", OUT_DIR)
print("This notebook does not treat analyst proxies as true revisions.")


## 2. Money2 与新增信息特征契约

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

CURRENT_MONEY_COLS = [
    "mf_main_mean5_rank", "mf_main_mean20_rank", "mf_main_accel_rank",
    "mf_main_positive_ratio20", "mf_xl_mean20_rank", "mf_big_small_spread20_rank",
    "mf_main_return_divergence", "mf_observation_days20",
    "mt_fin_value_chg5_rank", "mt_fin_value_chg20_rank", "mt_fin_buy_to_balance20_rank",
    "mt_observation_days20", "mt_is_marginable",
]
MONEY2_COLS = [
    "mq_main_std20_rank", "mq_main_ir20_rank", "mq_main_positive_ratio5_rank",
    "mq_main_sign_flip20_rank", "mq_main_slope20_rank", "mq_flow_return_corr20_rank",
    "mq_main_abs_concentration5_20_rank", "mq_big_flow_share20_rank",
    "mq_signed_price_efficiency20_rank", "mq_main_mean60_rank", "mq_main_accel5_60_rank",
    "mq_main20_rank_chg1m", "mq_main20_rank_mean3m", "mq_xl20_rank_chg1m",
    "mq_fin_balance_vs_mean60_rank", "mq_fin_balance_z60_rank",
    "mq_fin_buy_accel5_20_rank", "mq_fin_balance_up_ratio20_rank", "mq_fin20_rank_chg1m",
]
MONEY2_BASELINE_COLS = unique_keep_order(CURRENT_MONEY_COLS + MONEY2_COLS)

P2_FEATURES = [
    "ni_reduction_ratio180_rank", "ni_increase_ratio180_rank", "ni_net_increase_ratio180_rank",
    "ni_unlock_ratio30_rank", "ni_unlock_ratio90_rank",
]
P3_FEATURES = [
    "ni_np_yoy_rank", "ni_revenue_yoy_rank", "ni_np_yoy_change_rank",
    "ni_revenue_yoy_change_rank", "ni_report_age_rank", "ni_post_report_excess_rank",
    "ni_new_report_flag",
]
P4_FEATURES = [
    "ni_holder_count_change_rank", "ni_top10_concentration_rank",
    "ni_top10_concentration_change_rank", "ni_institution_share_rank",
    "ni_institution_share_change_rank", "ni_holder_data_age_rank",
]

feature_contract_df = pd.DataFrame([
    {"family": "money2_baseline", "count": len(MONEY2_BASELINE_COLS), "features": ",".join(MONEY2_BASELINE_COLS)},
    {"family": "p1_true_analyst_revision", "count": 0, "features": "requires external/detail source"},
    {"family": "p2_shareholder_unlock", "count": len(P2_FEATURES), "features": ",".join(P2_FEATURES)},
    {"family": "p3_actual_fundamental_reaction", "count": len(P3_FEATURES), "features": ",".join(P3_FEATURES)},
    {"family": "p4_holder_structure", "count": len(P4_FEATURES), "features": ",".join(P4_FEATURES)},
])
feature_contract_df.to_csv(OUT_DIR / "v98_feature_contract.csv", index=False)
display_df(feature_contract_df, 10)


## 3. 加载 V96 冻结 Money2 面板

In [ ]:
def resolve_panel_path():
    candidates = []
    if PANEL_PATH_OVERRIDE:
        candidates.append(Path(str(PANEL_PATH_OVERRIDE)))
    candidates.extend(PANEL_CANDIDATES)
    for root in [PROJECT_DIR, PROJECT_DIR.parent]:
        try:
            candidates.extend(list(root.glob("**/v96_money2_enriched_panel.csv")))
        except Exception:
            pass
    required = [STOCK_COL, DATE_COL, NEXT_DATE_COL, FEATURE_DATE_COL, TARGET_COL] + FULL_V46_COLS + MONEY2_BASELINE_COLS
    checked = []
    for candidate in candidates:
        path = Path(str(candidate))
        if not path.exists() or not path.is_file():
            continue
        try:
            header = pd.read_csv(str(path), nrows=0)
            missing = [c for c in required if c not in header.columns]
            checked.append({"path": str(path), "missing": ",".join(missing)})
            if not missing:
                return path, required, pd.DataFrame(checked)
        except Exception as exc:
            checked.append({"path": str(path), "missing": str(exc)})
    raise IOError("No valid V96 panel. Run V96 first or set PANEL_PATH_OVERRIDE. Checked: %s" % checked)


PANEL_PATH, PANEL_USECOLS, panel_path_audit_df = resolve_panel_path()
df_base = pd.read_csv(str(PANEL_PATH), usecols=PANEL_USECOLS, low_memory=False)
for col in [DATE_COL, NEXT_DATE_COL, FEATURE_DATE_COL]:
    df_base[col] = pd.to_datetime(df_base[col], errors="coerce")
df_base = df_base.dropna(subset=[STOCK_COL, DATE_COL, NEXT_DATE_COL, FEATURE_DATE_COL, TARGET_COL])
df_base = df_base.drop_duplicates([STOCK_COL, DATE_COL], keep="last").sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)

base_audit_df = pd.DataFrame([
    {"check": "rows", "value": len(df_base)},
    {"check": "months", "value": df_base[DATE_COL].nunique()},
    {"check": "stocks", "value": df_base[STOCK_COL].nunique()},
    {"check": "duplicate_stock_date", "value": int(df_base.duplicated([STOCK_COL, DATE_COL]).sum())},
    {"check": "feature_date_not_before_rebalance", "value": int((df_base[FEATURE_DATE_COL] >= df_base[DATE_COL]).sum())},
    {"check": "next_date_not_after_rebalance", "value": int((df_base[NEXT_DATE_COL] <= df_base[DATE_COL]).sum())},
])
panel_path_audit_df.to_csv(QA_DIR / "v98_panel_path_audit.csv", index=False)
base_audit_df.to_csv(QA_DIR / "v98_base_data_audit.csv", index=False)
display_df(base_audit_df, 20)


## 4. P1 准入与 P2/P4 标准表预检

In [ ]:
analyst_gate_df = pd.DataFrame([{
    "family": "p1_true_analyst_revision",
    "source_name": ANALYST_SOURCE_NAME or "not_configured",
    "required_fields": ",".join(ANALYST_REQUIRED_FIELDS),
    "ready": False,
    "reason": "No verified row-level analyst source. Static jqfactor consensus levels/rank changes are proxies and are excluded.",
}])

TABLE_SPECS = [
    {"family": "p2", "table": "STK_SHAREHOLDERS_SHARE_CHANGE", "purpose": "completed_holder_increase_decrease"},
    {"family": "p2", "table": "STK_LIMITED_SHARES_LIST", "purpose": "announced_future_unlock"},
    {"family": "p4", "table": "STK_HOLDER_NUM", "purpose": "holder_count"},
    {"family": "p4", "table": "STK_SHAREHOLDER_FLOATING_TOP10", "purpose": "top10_and_institution_concentration"},
]


def table_preflight(spec):
    row = dict(spec)
    row.update({"available": False, "columns": "", "detail": ""})
    try:
        table = getattr(finance, spec["table"])
        sample = finance.run_query(query(table).limit(1))
        columns = list(sample.columns) if sample is not None else []
        row.update({"available": True, "columns": ",".join(columns), "detail": "sample_ok"})
    except Exception as exc:
        row["detail"] = str(exc)
    return row


preflight_rows = []
for spec in progress_iter(TABLE_SPECS, total=len(TABLE_SPECS), desc="source preflight"):
    preflight_rows.append(table_preflight(spec))
source_preflight_df = pd.DataFrame(preflight_rows)
analyst_gate_df.to_csv(OUT_DIR / "v98_analyst_data_gate.csv", index=False)
source_preflight_df.to_csv(OUT_DIR / "v98_source_preflight.csv", index=False)
display_df(source_preflight_df, 20)


## 5. 获取 P2/P4 原始记录并缓存

In [ ]:
def raw_cache_path(table_name):
    return CACHE_DIR / ("%s_%s.csv" % (SOURCE_CACHE_VERSION, table_name.lower()))


def calendar_windows(start_date, end_date, window_years):
    start_date = pd.Timestamp(start_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()
    if window_years is None:
        return [(start_date, end_date)]
    windows = []
    current = start_date
    while current <= end_date:
        boundary = pd.Timestamp(year=current.year + int(window_years), month=1, day=1) - pd.Timedelta(days=1)
        window_end = _bi.min(boundary, end_date)
        windows.append((current, window_end))
        current = window_end + pd.Timedelta(days=1)
    return windows


def compact_finance_part(table_name, part):
    if part is None or not len(part):
        return pd.DataFrame()
    required = TABLE_REQUIRED_FIELDS.get(table_name, [])
    keep = [col for col in required if col in part.columns]
    compact = part[keep].copy() if keep else part.copy()
    if table_name != "STK_SHAREHOLDER_FLOATING_TOP10":
        return compact.drop_duplicates().reset_index(drop=True)

    group_cols = [col for col in ["code", "end_date", "pub_date"] if col in compact.columns]
    if len(group_cols) < 3 or "share_ratio" not in compact.columns:
        return compact.drop_duplicates().reset_index(drop=True)
    ratio = pd.to_numeric(compact["share_ratio"], errors="coerce").fillna(0.0)
    classifier = pd.Series("", index=compact.index)
    for col in ["shareholder_class", "shareholder_name"]:
        if col in compact.columns:
            classifier = classifier + " " + compact[col].fillna("").astype(str)
    classifier = classifier.str.upper()
    institution = pd.Series(False, index=compact.index)
    for keyword in INSTITUTION_KEYWORDS:
        institution = institution | classifier.str.contains(str(keyword).upper(), na=False)
    compact["report_share_ratio"] = ratio.astype(np.float32)
    compact["report_institution_ratio"] = ratio.where(institution, 0.0).astype(np.float32)
    compact = compact.groupby(group_cols, as_index=False)[
        ["report_share_ratio", "report_institution_ratio"]
    ].sum()
    return compact.reset_index(drop=True)


def fetch_finance_table_history(table_name, stocks, start_pub_date, end_pub_date):
    path = raw_cache_path(table_name)
    if path.exists() and not FORCE_REFETCH:
        cached = pd.read_csv(str(path), low_memory=False)
        return cached, {"table": table_name, "status": "cache", "rows": len(cached), "detail": ""}
    try:
        table = getattr(finance, table_name)
    except Exception as exc:
        return pd.DataFrame(), {"table": table_name, "status": "unavailable", "rows": 0, "detail": str(exc)}
    policy = TABLE_FETCH_POLICIES.get(table_name, {"stock_batch_size": SOURCE_BATCH_SIZE, "window_years": None})
    stock_batch_size = int(policy.get("stock_batch_size", SOURCE_BATCH_SIZE))
    windows = calendar_windows(start_pub_date, end_pub_date, policy.get("window_years"))
    rows = []
    errors = []
    max_chunk_rows = 0
    truncated_chunks = 0
    raw_rows_before_compaction = 0
    tasks = [(batch, window_start, window_end) for batch in chunks(stocks, stock_batch_size) for window_start, window_end in windows]
    for batch, window_start, window_end in progress_iter(tasks, total=len(tasks), desc="fetch %s" % table_name, leave=False):
        try:
            requested_fields = [
                getattr(table, field_name) for field_name in TABLE_REQUIRED_FIELDS.get(table_name, [])
                if hasattr(table, field_name)
            ]
            q = query(*(requested_fields or [table])).filter(
                table.code.in_(batch),
                table.pub_date >= pd.Timestamp(window_start).date(),
                table.pub_date <= pd.Timestamp(window_end).date(),
            )
            if hasattr(finance, "run_offset_query"):
                part = finance.run_offset_query(q)
            else:
                part = finance.run_query(q)
            if part is not None and len(part):
                raw_rows_before_compaction += int(len(part))
                max_chunk_rows = _bi.max(max_chunk_rows, int(len(part)))
                if int(len(part)) > int(MAX_SAFE_CHUNK_ROWS):
                    truncated_chunks += 1
                    errors.append("possible row-limit truncation %s %s..%s rows=%s" % (table_name, window_start.date(), window_end.date(), len(part)))
                compact = compact_finance_part(table_name, part)
                if len(compact):
                    rows.append(compact)
                del part, compact
                gc.collect()
        except Exception as exc:
            errors.append(str(exc))
    raw = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()
    if len(raw):
        raw = raw.drop_duplicates().reset_index(drop=True)
        raw.to_csv(str(path), index=False)
    status = "ok" if not errors else ("partial" if len(raw) else "error")
    return raw, {
        "table": table_name, "status": status, "rows": len(raw),
        "max_chunk_rows": max_chunk_rows, "truncated_chunks": truncated_chunks,
        "stock_batch_size": stock_batch_size, "window_count": len(windows),
        "raw_rows_before_compaction": raw_rows_before_compaction,
        "compacted_rows": len(raw),
        "detail": " | ".join(errors[:3]),
    }


all_stocks = _bi.sorted(df_base[STOCK_COL].dropna().astype(str).unique().tolist())
raw_start = df_base[FEATURE_DATE_COL].min() - pd.DateOffset(years=4)
raw_end = df_base[FEATURE_DATE_COL].max()
raw_tables = {}
raw_status_rows = []
for spec in TABLE_SPECS:
    raw, status = fetch_finance_table_history(spec["table"], all_stocks, raw_start, raw_end)
    raw_tables[spec["table"]] = raw
    status.update({"family": spec["family"], "purpose": spec["purpose"]})
    raw_status_rows.append(status)
    gc.collect()
raw_status_df = pd.DataFrame(raw_status_rows)
raw_status_df.to_csv(QA_DIR / "v98_raw_source_status.csv", index=False)
display_df(raw_status_df, 20)


## 6. 构造逐月 P2/P3/P4 时点快照

In [ ]:
def first_existing(columns, candidates):
    for col in candidates:
        if col in columns:
            return col
    return None


def numeric_col(df, candidates):
    col = first_existing(df.columns, candidates)
    if col is None:
        return pd.Series(np.nan, index=df.index), None
    return pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan), col


def date_col(df, candidates):
    col = first_existing(df.columns, candidates)
    if col is None:
        return pd.Series(pd.NaT, index=df.index), None
    return pd.to_datetime(df[col], errors="coerce"), col


def stock_frame(stocks, rebalance_date, feature_date):
    return pd.DataFrame({
        STOCK_COL: list(stocks), DATE_COL: pd.Timestamp(rebalance_date), FEATURE_DATE_COL: pd.Timestamp(feature_date),
    })


def prepare_known(raw, feature_date, lookback_days=None):
    if raw is None or not len(raw):
        return pd.DataFrame()
    pub, pub_col = date_col(raw, ["pub_date", "pubDate", "announcement_date"])
    if pub_col is None:
        return pd.DataFrame()
    mask = pub.notnull() & (pub <= pd.Timestamp(feature_date))
    if lookback_days is not None:
        mask &= pub >= pd.Timestamp(feature_date) - pd.Timedelta(days=int(lookback_days))
    result = raw.loc[mask].copy()
    result["_pub_date"] = pub.loc[mask].values
    return result


def build_p2_snapshot(stocks, rebalance_date, feature_date):
    out = stock_frame(stocks, rebalance_date, feature_date)
    out["ni_reduction_ratio180"] = 0.0
    out["ni_increase_ratio180"] = 0.0
    out["ni_unlock_ratio30"] = 0.0
    out["ni_unlock_ratio90"] = 0.0
    max_pub = pd.Series(pd.NaT, index=out.index)

    changes = prepare_known(raw_tables.get("STK_SHAREHOLDERS_SHARE_CHANGE"), feature_date, 180)
    if len(changes):
        ratio, ratio_col = numeric_col(changes, ["change_ratio", "changeRatio"])
        type_col = first_existing(changes.columns, ["type", "change_type", "change_direction"])
        changes["_ratio"] = ratio.abs().fillna(0)
        type_text = changes[type_col].astype(str) if type_col else pd.Series("", index=changes.index)
        reduce_mask = type_text.str.contains("减", na=False) | type_text.isin(["1", "1.0"])
        changes["_reduction"] = changes["_ratio"].where(reduce_mask, 0)
        changes["_increase"] = changes["_ratio"].where(~reduce_mask, 0)
        grouped = changes.groupby("code")[["_reduction", "_increase"]].sum()
        out["ni_reduction_ratio180"] = out[STOCK_COL].map(grouped["_reduction"]).fillna(0)
        out["ni_increase_ratio180"] = out[STOCK_COL].map(grouped["_increase"]).fillna(0)

    unlock = prepare_known(raw_tables.get("STK_LIMITED_SHARES_LIST"), feature_date, None)
    if len(unlock):
        unlock_date, unlock_date_col = date_col(unlock, ["expected_unlimited_date", "actual_unlimited_date"])
        ratio, ratio_col = numeric_col(unlock, ["expected_unlimited_ratio", "actual_unlimited_ratio"])
        unlock["_unlock_date"] = unlock_date
        unlock["_ratio"] = ratio.fillna(0).clip(lower=0)
        future = unlock[unlock["_unlock_date"] > pd.Timestamp(feature_date)].copy()
        for days, output_col in [(30, "ni_unlock_ratio30"), (90, "ni_unlock_ratio90")]:
            within = future[future["_unlock_date"] <= pd.Timestamp(feature_date) + pd.Timedelta(days=days)]
            grouped = within.groupby("code")["_ratio"].sum() if len(within) else pd.Series(dtype=float)
            out[output_col] = out[STOCK_COL].map(grouped).fillna(0)
    out["ni_net_increase_ratio180"] = out["ni_increase_ratio180"] - out["ni_reduction_ratio180"]
    known_parts = [x for x in [changes, unlock] if len(x)]
    if known_parts:
        pub_map = pd.concat(known_parts, ignore_index=True, sort=False).groupby("code")["_pub_date"].max()
        out["p2_max_pub_date_seen"] = out[STOCK_COL].map(pub_map)
    else:
        out["p2_max_pub_date_seen"] = pd.NaT
    return out


def institution_mask(values):
    text = values.fillna("").astype(str).str.upper()
    mask = pd.Series(False, index=text.index)
    for keyword in INSTITUTION_KEYWORDS:
        mask = mask | text.str.contains(str(keyword).upper(), na=False)
    return mask


def build_p4_snapshot(stocks, rebalance_date, feature_date):
    out = stock_frame(stocks, rebalance_date, feature_date)
    holder = prepare_known(raw_tables.get("STK_HOLDER_NUM"), feature_date, None)
    if len(holder):
        end, end_col = date_col(holder, ["end_date", "stat_date", "statDate"])
        count, count_col = numeric_col(holder, ["share_holders", "a_share_holders"])
        holder["_end_date"] = end
        holder["_count"] = count
        holder = holder.dropna(subset=["code", "_end_date", "_count"])
        holder = holder.sort_values(["code", "_end_date", "_pub_date"]).drop_duplicates(["code", "_end_date"], keep="last")
        latest_rows = []
        for stock, part in holder.groupby("code"):
            part = part.sort_values(["_end_date", "_pub_date"])
            latest = part.iloc[-1]
            previous = part.iloc[-2] if len(part) >= 2 else None
            latest_count = float(latest["_count"])
            previous_count = float(previous["_count"]) if previous is not None else np.nan
            change = latest_count / previous_count - 1.0 if previous_count > 0 else np.nan
            latest_rows.append({
                STOCK_COL: stock, "ni_holder_count_change": change,
                "ni_holder_data_age": (pd.Timestamp(feature_date) - latest["_pub_date"]).days,
                "p4_holder_max_pub": latest["_pub_date"],
            })
        holder_snapshot = pd.DataFrame(latest_rows)
        out = out.merge(holder_snapshot, on=STOCK_COL, how="left")
    else:
        out["ni_holder_count_change"] = np.nan
        out["ni_holder_data_age"] = np.nan
        out["p4_holder_max_pub"] = pd.NaT

    top10 = prepare_known(raw_tables.get("STK_SHAREHOLDER_FLOATING_TOP10"), feature_date, None)
    if len(top10):
        end, end_col = date_col(top10, ["end_date", "stat_date", "statDate"])
        top10["_end_date"] = end
        if "report_share_ratio" in top10.columns and "report_institution_ratio" in top10.columns:
            top10["_ratio"] = pd.to_numeric(top10["report_share_ratio"], errors="coerce").fillna(0)
            top10["_institution_ratio"] = pd.to_numeric(top10["report_institution_ratio"], errors="coerce").fillna(0)
            report = top10.dropna(subset=["code", "_end_date"]).sort_values(
                ["code", "_end_date", "_pub_date"]
            ).drop_duplicates(["code", "_end_date"], keep="last")
            report = report[["code", "_end_date", "_ratio", "_institution_ratio", "_pub_date"]].copy()
        else:
            ratio, ratio_col = numeric_col(top10, ["share_ratio", "holding_ratio"])
            name_col = first_existing(top10.columns, ["shareholder_name", "holder_name"])
            class_col = first_existing(top10.columns, ["shareholder_class", "shareholder_type", "share_snature"])
            classifier = pd.Series("", index=top10.index)
            if class_col:
                classifier = classifier + " " + top10[class_col].fillna("").astype(str)
            if name_col:
                classifier = classifier + " " + top10[name_col].fillna("").astype(str)
            top10["_ratio"] = ratio.fillna(0)
            top10["_institution_ratio"] = top10["_ratio"].where(institution_mask(classifier), 0)
            top10 = top10.dropna(subset=["code", "_end_date"])
            report = top10.groupby(["code", "_end_date"]).agg({
                "_ratio": "sum", "_institution_ratio": "sum", "_pub_date": "max",
            }).reset_index()
        latest_rows = []
        for stock, part in report.groupby("code"):
            part = part.sort_values(["_end_date", "_pub_date"])
            latest = part.iloc[-1]
            previous = part.iloc[-2] if len(part) >= 2 else None
            latest_rows.append({
                STOCK_COL: stock,
                "ni_top10_concentration": float(latest["_ratio"]),
                "ni_top10_concentration_change": float(latest["_ratio"] - previous["_ratio"]) if previous is not None else np.nan,
                "ni_institution_share": float(latest["_institution_ratio"]),
                "ni_institution_share_change": float(latest["_institution_ratio"] - previous["_institution_ratio"]) if previous is not None else np.nan,
                "p4_top10_max_pub": latest["_pub_date"],
            })
        top10_snapshot = pd.DataFrame(latest_rows)
        out = out.merge(top10_snapshot, on=STOCK_COL, how="left")
    else:
        for col in ["ni_top10_concentration", "ni_top10_concentration_change", "ni_institution_share", "ni_institution_share_change"]:
            out[col] = np.nan
        out["p4_top10_max_pub"] = pd.NaT
    return out


def fetch_fundamental_snapshot(stocks, rebalance_date, feature_date):
    path = CACHE_DIR / ("fundamental_%s.csv" % pd.Timestamp(rebalance_date).strftime("%Y%m%d"))
    if path.exists() and not FORCE_REFETCH:
        return pd.read_csv(str(path), low_memory=False)
    rows = []
    for batch in chunks(stocks, SOURCE_BATCH_SIZE):
        try:
            fields = [income.code, income.pubDate, income.statDate]
            for field_name in ["inc_net_profit_year_on_year", "inc_revenue_year_on_year"]:
                if hasattr(indicator, field_name):
                    fields.append(getattr(indicator, field_name))
            q = query(*fields).filter(income.code.in_(batch))
            part = get_fundamentals(q, date=pd.Timestamp(feature_date).date())
            if part is not None and len(part):
                rows.append(part)
        except Exception as exc:
            print("fundamental batch error:", str(exc)[:200])
    raw = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()
    out = stock_frame(stocks, rebalance_date, feature_date)
    if len(raw):
        raw = raw.rename(columns={
            "code": STOCK_COL, "pubDate": "ni_report_pub_date", "statDate": "ni_report_stat_date",
            "inc_net_profit_year_on_year": "ni_np_yoy",
            "inc_revenue_year_on_year": "ni_revenue_yoy",
        })
        keep = [c for c in [STOCK_COL, "ni_report_pub_date", "ni_report_stat_date", "ni_np_yoy", "ni_revenue_yoy"] if c in raw.columns]
        raw = raw[keep].drop_duplicates(STOCK_COL, keep="last")
        out = out.merge(raw, on=STOCK_COL, how="left")
    for col in ["ni_report_pub_date", "ni_report_stat_date"]:
        if col not in out.columns:
            out[col] = pd.NaT
        out[col] = pd.to_datetime(out[col], errors="coerce")
    for col in ["ni_np_yoy", "ni_revenue_yoy"]:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    out["ni_report_age"] = (pd.Timestamp(feature_date) - out["ni_report_pub_date"]).dt.days
    out["ni_post_report_excess"] = np.nan

    recent = out[(out["ni_report_age"] >= 0) & (out["ni_report_age"] <= 35)][[STOCK_COL, "ni_report_pub_date"]].dropna()
    if len(recent):
        start_date = recent["ni_report_pub_date"].min() - pd.Timedelta(days=3)
        try:
            price = get_price(
                recent[STOCK_COL].tolist(), start_date=start_date, end_date=pd.Timestamp(feature_date),
                frequency="daily", fields=["close"], skip_paused=False, fq="pre", panel=False,
            )
            benchmark = get_price(
                "000906.XSHG", start_date=start_date, end_date=pd.Timestamp(feature_date),
                frequency="daily", fields=["close"], skip_paused=False, fq="pre",
            )
            if price is not None and len(price):
                price = price.reset_index() if STOCK_COL not in price.columns and "code" not in price.columns else price.copy()
                code_col = first_existing(price.columns, ["code", "security", STOCK_COL])
                time_col = first_existing(price.columns, ["time", "date", "index"])
                if code_col and time_col and "close" in price.columns:
                    price[time_col] = pd.to_datetime(price[time_col], errors="coerce")
                    benchmark = benchmark.copy()
                    benchmark.index = pd.to_datetime(benchmark.index)
                    excess_map = {}
                    pub_map = recent.set_index(STOCK_COL)["ni_report_pub_date"].to_dict()
                    for stock, part in price.groupby(code_col):
                        pub_value = pub_map.get(stock)
                        if pd.isnull(pub_value):
                            continue
                        pub_date = pd.Timestamp(pub_value)
                        part = part.sort_values(time_col)
                        part = part[(part[time_col] >= pub_date) & (part[time_col] <= pd.Timestamp(feature_date))]
                        if len(part) >= 2:
                            start_trade = part[time_col].iloc[0]
                            stock_ret = float(part["close"].iloc[-1] / part["close"].iloc[0] - 1.0)
                            bench_part = benchmark[(benchmark.index >= start_trade) & (benchmark.index <= pd.Timestamp(feature_date))]
                            bench_ret = float(bench_part["close"].iloc[-1] / bench_part["close"].iloc[0] - 1.0) if len(bench_part) >= 2 else 0.0
                            excess_map[stock] = stock_ret - bench_ret
                    out["ni_post_report_excess"] = out[STOCK_COL].map(excess_map)
        except Exception as exc:
            print("post-report reaction unavailable:", str(exc)[:200])
    out.to_csv(str(path), index=False)
    return out


snapshot_rows = []
snapshot_status_rows = []
date_plan = df_base[[DATE_COL, FEATURE_DATE_COL]].drop_duplicates().sort_values(DATE_COL)
if SMOKE_MODE:
    date_plan = date_plan.tail(SMOKE_MAX_MONTHS)
for _, date_row in progress_iter(list(date_plan.iterrows()), total=len(date_plan), desc="build P2-P4 snapshots"):
    rebalance_date = pd.Timestamp(date_row[DATE_COL])
    feature_date = pd.Timestamp(date_row[FEATURE_DATE_COL])
    stocks = df_base.loc[df_base[DATE_COL] == rebalance_date, STOCK_COL].astype(str).tolist()
    parts = []
    for family, builder in [("p2", build_p2_snapshot), ("p4", build_p4_snapshot)]:
        try:
            part = builder(stocks, rebalance_date, feature_date)
            parts.append(part)
            snapshot_status_rows.append({"family": family, DATE_COL: rebalance_date, "status": "ok", "rows": len(part), "detail": ""})
        except Exception as exc:
            snapshot_status_rows.append({"family": family, DATE_COL: rebalance_date, "status": "error", "rows": 0, "detail": str(exc)})
    try:
        p3 = fetch_fundamental_snapshot(stocks, rebalance_date, feature_date)
        parts.append(p3)
        snapshot_status_rows.append({"family": "p3", DATE_COL: rebalance_date, "status": "ok", "rows": len(p3), "detail": ""})
    except Exception as exc:
        snapshot_status_rows.append({"family": "p3", DATE_COL: rebalance_date, "status": "error", "rows": 0, "detail": str(exc)})
    merged = stock_frame(stocks, rebalance_date, feature_date)
    for part in parts:
        add_cols = [c for c in part.columns if c not in [DATE_COL, FEATURE_DATE_COL]]
        merged = merged.merge(part[add_cols], on=STOCK_COL, how="left")
    snapshot_rows.append(merged)
    del parts, merged
    gc.collect()

new_info_raw_df = pd.concat(snapshot_rows, ignore_index=True, sort=False) if snapshot_rows else pd.DataFrame()
del snapshot_rows
raw_tables.clear()
del raw_tables
gc.collect()
snapshot_status_df = pd.DataFrame(snapshot_status_rows)
snapshot_status_df.to_csv(QA_DIR / "v98_monthly_snapshot_status.csv", index=False)


## 7. 时点审计、稳健截面排名与数据准入

In [ ]:
def safe_rank_by_month(frame, raw_col, rank_col):
    result = pd.Series(np.nan, index=frame.index, dtype=float)
    for date_value, indices in frame.groupby(DATE_COL).groups.items():
        values = pd.to_numeric(frame.loc[indices, raw_col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = values.notnull()
        if int(valid.sum()) > 0:
            result.loc[values.index[valid]] = values[valid].rank(method="average", pct=True).values
    frame[rank_col] = result.astype(np.float32)


new_info_raw_df = new_info_raw_df.sort_values([STOCK_COL, DATE_COL]).reset_index(drop=True)
for col in ["ni_np_yoy", "ni_revenue_yoy"]:
    previous = new_info_raw_df.groupby(STOCK_COL)[col].shift(1)
    new_info_raw_df[col + "_change"] = pd.to_numeric(new_info_raw_df[col], errors="coerce") - pd.to_numeric(previous, errors="coerce")
previous_stat = new_info_raw_df.groupby(STOCK_COL)["ni_report_stat_date"].shift(1)
new_info_raw_df["ni_new_report_flag"] = (
    new_info_raw_df["ni_report_stat_date"].notnull() &
    (new_info_raw_df["ni_report_stat_date"] != previous_stat)
).astype(np.int8)

rank_map = {
    "ni_reduction_ratio180": "ni_reduction_ratio180_rank",
    "ni_increase_ratio180": "ni_increase_ratio180_rank",
    "ni_net_increase_ratio180": "ni_net_increase_ratio180_rank",
    "ni_unlock_ratio30": "ni_unlock_ratio30_rank",
    "ni_unlock_ratio90": "ni_unlock_ratio90_rank",
    "ni_np_yoy": "ni_np_yoy_rank",
    "ni_revenue_yoy": "ni_revenue_yoy_rank",
    "ni_np_yoy_change": "ni_np_yoy_change_rank",
    "ni_revenue_yoy_change": "ni_revenue_yoy_change_rank",
    "ni_report_age": "ni_report_age_rank",
    "ni_post_report_excess": "ni_post_report_excess_rank",
    "ni_holder_count_change": "ni_holder_count_change_rank",
    "ni_top10_concentration": "ni_top10_concentration_rank",
    "ni_top10_concentration_change": "ni_top10_concentration_change_rank",
    "ni_institution_share": "ni_institution_share_rank",
    "ni_institution_share_change": "ni_institution_share_change_rank",
    "ni_holder_data_age": "ni_holder_data_age_rank",
}
for raw_col, rank_col in progress_iter(list(rank_map.items()), total=len(rank_map), desc="safe cross-section ranks"):
    if raw_col not in new_info_raw_df.columns:
        new_info_raw_df[raw_col] = np.nan
    safe_rank_by_month(new_info_raw_df, raw_col, rank_col)

for col in ["p2_max_pub_date_seen", "p4_holder_max_pub", "p4_top10_max_pub", "ni_report_pub_date"]:
    if col not in new_info_raw_df.columns:
        new_info_raw_df[col] = pd.NaT
    new_info_raw_df[col] = pd.to_datetime(new_info_raw_df[col], errors="coerce")

pit_rows = []
for col, family in [
    ("p2_max_pub_date_seen", "p2"), ("ni_report_pub_date", "p3"),
    ("p4_holder_max_pub", "p4_holder"), ("p4_top10_max_pub", "p4_top10"),
]:
    violations = int((new_info_raw_df[col].notnull() & (new_info_raw_df[col] > new_info_raw_df[FEATURE_DATE_COL])).sum())
    pit_rows.append({"family": family, "date_column": col, "violations": violations, "status": "ok" if violations == 0 else "fail"})
pit_audit_df = pd.DataFrame(pit_rows)

new_info_raw_df = new_info_raw_df.drop(columns=[FEATURE_DATE_COL], errors="ignore")
df_all = df_base.merge(new_info_raw_df, on=[STOCK_COL, DATE_COL], how="left")

family_features = {
    "p2_shareholder_unlock": P2_FEATURES,
    "p3_actual_fundamental_reaction": P3_FEATURES,
    "p4_holder_structure": P4_FEATURES,
}
coverage_anchors = {
    "p2_shareholder_unlock": P2_FEATURES,
    "p3_actual_fundamental_reaction": ["ni_np_yoy_rank", "ni_revenue_yoy_rank"],
    "p4_holder_structure": ["ni_holder_count_change_rank", "ni_top10_concentration_rank"],
}
good_statuses = set(["ok", "cache"])
raw_status_map = dict(zip(raw_status_df["table"], raw_status_df["status"])) if len(raw_status_df) else {}
source_ok_map = {
    "p2_shareholder_unlock": _bi.all([
        raw_status_map.get("STK_SHAREHOLDERS_SHARE_CHANGE") in good_statuses,
        raw_status_map.get("STK_LIMITED_SHARES_LIST") in good_statuses,
        int(raw_status_df.loc[raw_status_df["table"] == "STK_SHAREHOLDERS_SHARE_CHANGE", "rows"].sum()) > 0,
        int(raw_status_df.loc[raw_status_df["table"] == "STK_LIMITED_SHARES_LIST", "rows"].sum()) > 0,
    ]),
    "p3_actual_fundamental_reaction": bool(len(snapshot_status_df[snapshot_status_df["family"] == "p3"])) and not bool((snapshot_status_df[snapshot_status_df["family"] == "p3"]["status"] == "error").any()),
    "p4_holder_structure": _bi.all([
        raw_status_map.get("STK_HOLDER_NUM") in good_statuses,
        raw_status_map.get("STK_SHAREHOLDER_FLOATING_TOP10") in good_statuses,
        int(raw_status_df.loc[raw_status_df["table"] == "STK_HOLDER_NUM", "rows"].sum()) > 0,
        int(raw_status_df.loc[raw_status_df["table"] == "STK_SHAREHOLDER_FLOATING_TOP10", "rows"].sum()) > 0,
    ]),
}
coverage_rows = []
for family, features in family_features.items():
    existing = [c for c in features if c in df_all.columns]
    anchors = [c for c in coverage_anchors[family] if c in df_all.columns]
    row_coverage = float(df_all[anchors].notnull().any(axis=1).mean()) if anchors else 0.0
    month_coverage = df_all.groupby(DATE_COL)[anchors].apply(lambda x: float(x.notnull().any(axis=1).mean())) if anchors else pd.Series(dtype=float)
    source_ok = bool(source_ok_map.get(family, False))
    coverage_rows.append({
        "family": family, "feature_count": len(existing), "row_coverage": row_coverage,
        "min_month_coverage": float(month_coverage.min()) if len(month_coverage) else 0.0,
        "months": len(month_coverage), "source_query_ok": source_ok,
        "data_gate": bool(source_ok and row_coverage >= MIN_SOURCE_COVERAGE),
    })
coverage_df = pd.DataFrame(coverage_rows)

pit_audit_df.to_csv(QA_DIR / "v98_point_in_time_audit.csv", index=False)
coverage_df.to_csv(OUT_DIR / "v98_family_source_coverage.csv", index=False)
df_all.to_csv(OUT_DIR / "v98_external_information_enriched_panel.csv", index=False)
display_df(pit_audit_df, 20)
display_df(coverage_df, 20)
del df_base, new_info_raw_df
gc.collect()


## 8. 兼容聚宽旧环境的训练与 Top8 指标函数

In [ ]:
def numeric_series(df, col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)


def corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    cols = [c for c in feature_cols if c in train_df.columns]
    corr = train_df[cols].corr()
    graph = defaultdict(list)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            value = corr.iloc[i, j]
            if not pd.isnull(value) and abs(value) > threshold:
                graph[cols[i]].append(cols[j])
                graph[cols[j]].append(cols[i])
    visited = set()
    components = []
    for col in cols:
        if col in visited:
            continue
        stack = [col]
        component = []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(graph[current])
        components.append(component)
    return components


def select_v46_features(train_df):
    missing_map = train_df[FULL_V46_COLS].isnull().sum().to_dict()
    keep = []
    removed = []
    for component in corr_components(train_df, FULL_V46_COLS, CORR_THRESHOLD_V46):
        ordered = _bi.sorted(component, key=lambda x: (missing_map.get(x, 10 ** 12), x))
        keep.append(ordered[0])
        removed.extend(ordered[1:])
    return unique_keep_order(keep), unique_keep_order(removed)


def select_incremental_features(train_df, candidate_cols):
    usable = []
    removed = []
    coverage_map = {}
    for col in candidate_cols:
        values = numeric_series(train_df, col)
        coverage = float(values.notnull().mean())
        coverage_map[col] = coverage
        if coverage >= MIN_INCREMENTAL_TRAIN_COVERAGE and values.nunique(dropna=True) > 1:
            usable.append(col)
        else:
            removed.append(col)
    if len(usable) <= 1:
        return unique_keep_order(usable), unique_keep_order(removed), coverage_map
    missing_map = train_df[usable].isnull().sum().to_dict()
    keep = []
    for component in corr_components(train_df, usable, CORR_THRESHOLD_INCREMENTAL):
        ordered = _bi.sorted(component, key=lambda x: (missing_map.get(x, 10 ** 12), x))
        keep.append(ordered[0])
        removed.extend(ordered[1:])
    return unique_keep_order(keep), unique_keep_order(removed), coverage_map


def model_params(seed, feature_fraction):
    params = dict(BASE_PARAMS)
    params.update({
        "feature_fraction": float(feature_fraction),
        "seed": int(seed), "feature_fraction_seed": int(seed),
        "bagging_seed": int(seed), "data_random_seed": int(seed),
        "num_threads": int(NUM_THREADS),
    })
    return params


def train_start_for_policy(cutoff, policy):
    cutoff = pd.Timestamp(cutoff)
    if policy == "expanding":
        return df_all[DATE_COL].min()
    if policy == "rolling60":
        return cutoff - pd.DateOffset(months=60) + pd.Timedelta(days=1)
    raise ValueError("unknown train policy: %s" % policy)


def build_train_test(cutoff, policy, eval_start=None, eval_end=None):
    cutoff = pd.Timestamp(cutoff)
    train_start = train_start_for_policy(cutoff, policy)
    train_mask = (
        (df_all[DATE_COL] >= train_start) &
        (df_all[DATE_COL] <= cutoff) &
        (df_all[NEXT_DATE_COL] <= cutoff)
    )
    if eval_start is None:
        eval_start = cutoff + pd.Timedelta(days=1)
    if eval_end is None:
        eval_end = cutoff + pd.DateOffset(months=12)
    test_mask = (df_all[DATE_COL] >= pd.Timestamp(eval_start)) & (df_all[DATE_COL] <= pd.Timestamp(eval_end))
    train = df_all.loc[train_mask]
    test = df_all.loc[test_mask]
    if SMOKE_MODE and test[DATE_COL].nunique() > SMOKE_MAX_MONTHS:
        keep_dates = _bi.sorted(test[DATE_COL].dropna().unique())[:SMOKE_MAX_MONTHS]
        test = test[test[DATE_COL].isin(keep_dates)]
    return train, test, pd.Timestamp(train_start)


def prepare_dataset(train_df, test_df, feature_cols):
    fill_values = train_df[feature_cols].median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
    y_train = train_df[TARGET_COL].astype(float).values
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
    dataset = lgb.Dataset(X_train, label=y_train, feature_name=list(feature_cols), free_raw_data=True)
    return dataset, X_test, fill_values


def train_predict_many_seeds(train_df, test_df, feature_cols, seeds, feature_fraction, meta):
    dataset, X_test, fill_values = prepare_dataset(train_df, test_df, feature_cols)
    score_map = {}
    model_rows = []
    importance_rows = []
    for seed in progress_iter(seeds, total=len(seeds), desc="seeds %s" % meta.get("family", "model"), leave=False):
        model = lgb.train(model_params(seed, feature_fraction), dataset, num_boost_round=int(FIXED_ITER))
        score_map[int(seed)] = np.asarray(model.predict(X_test, num_iteration=FIXED_ITER)).reshape(-1).astype(np.float32)
        gain = np.asarray(model.feature_importance(importance_type="gain"), dtype=float)
        split = np.asarray(model.feature_importance(importance_type="split"), dtype=float)
        model_row = dict(meta)
        model_row.update({
            "seed": int(seed), "feature_fraction": float(feature_fraction),
            "train_rows": len(train_df), "train_months": train_df[DATE_COL].nunique(),
            "test_rows": len(test_df), "test_months": test_df[DATE_COL].nunique(),
            "feature_count": len(feature_cols), "feature_cols": ",".join(feature_cols),
        })
        model_rows.append(model_row)
        for idx, feature in enumerate(feature_cols):
            imp_row = dict(meta)
            imp_row.update({
                "seed": int(seed), "feature_fraction": float(feature_fraction),
                "feature": feature, "importance_gain": float(gain[idx]),
                "importance_split": float(split[idx]),
            })
            importance_rows.append(imp_row)
        del model
        gc.collect()
    del dataset, X_test
    gc.collect()
    return score_map, model_rows, importance_rows, fill_values


def safe_rank_ic(y_true, score):
    y = pd.Series(np.asarray(y_true, dtype=float))
    s = pd.Series(np.asarray(score, dtype=float))
    valid = y.notnull() & s.notnull() & np.isfinite(y) & np.isfinite(s)
    if int(valid.sum()) < 5:
        return np.nan
    return float(y[valid].rank(method="average").corr(s[valid].rank(method="average")))


def binary_rank_auc(y_true, score):
    y = np.asarray(y_true).astype(int)
    s = pd.Series(np.asarray(score, dtype=float))
    valid = np.isfinite(s.values)
    y = y[valid]
    s = s[valid]
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = s.rank(method="average").values
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg))


def binary_ndcg_at_k(y_true, score, k):
    y = np.asarray(y_true).astype(float)
    s = np.asarray(score).astype(float)
    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]
    if len(y) == 0:
        return np.nan
    k = _bi.min(int(k), len(y))
    order = np.argsort(-s)[:k]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = float((y[order] * discounts).sum())
    ideal = np.sort(y)[::-1][:k]
    idcg = float((ideal * discounts).sum())
    return dcg / idcg if idcg > 0 else np.nan


def binary_map_at_k(y_true, score, k):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)
    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]
    if len(y) == 0:
        return np.nan
    k = _bi.min(int(k), len(y))
    order = np.argsort(-s)[:k]
    rel = y[order]
    total_rel = int(y.sum())
    denom = _bi.min(total_rel, k)
    if denom <= 0:
        return np.nan
    precision_sum = 0.0
    hits = 0
    for idx in range(k):
        if rel[idx] > 0:
            hits += 1
            precision_sum += hits / float(idx + 1)
    return precision_sum / float(denom)


def stock_board(stock):
    code_value = str(stock).split(".")[0]
    if code_value.startswith(("300", "301")):
        return "chinext"
    if code_value.startswith(("688", "689")):
        return "star"
    return "other"


def select_top8_board_capped(month_df, score_col):
    selected = []
    board_counts = {"chinext": 0, "star": 0}
    ranked = month_df.sort_values(score_col, ascending=False)
    for idx, row in ranked.iterrows():
        board = stock_board(row[STOCK_COL])
        if board in BOARD_CAPS and board_counts[board] >= BOARD_CAPS[board]:
            continue
        selected.append(idx)
        if board in board_counts:
            board_counts[board] += 1
        if len(selected) >= PRED_TOP_K:
            break
    return selected


def cross_section_rank(values):
    series = pd.Series(np.asarray(values, dtype=float))
    return series.rank(method="average", pct=True).fillna(0.5).values.astype(np.float32)


def evaluate_score_panel(panel, score_cols, common_meta, store_selected=False):
    metric_rows = []
    selected_rows = []
    previous_sets = dict((variant, None) for variant in score_cols)
    grouped = list(panel.groupby(DATE_COL))
    for date_value, month_raw in grouped:
        for variant, score_col in score_cols.items():
            month = month_raw[[STOCK_COL, DATE_COL, TARGET_COL, score_col]].copy()
            month = month.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET_COL, score_col])
            n = len(month)
            if n < 30:
                continue
            true_n = _bi.min(TRUE_TOP_N, n)
            month["true_rank"] = month[TARGET_COL].rank(method="first", ascending=False)
            binary_true = (month["true_rank"] <= true_n).astype(int).values
            selected_indices = select_top8_board_capped(month, score_col)
            selected = month.loc[selected_indices].copy()
            selected_set = set(selected[STOCK_COL].tolist())
            previous_set = previous_sets.get(variant)
            turnover = np.nan if previous_set is None else 1.0 - len(previous_set.intersection(selected_set)) / float(PRED_TOP_K)
            previous_sets[variant] = selected_set
            hits = int((selected["true_rank"] <= true_n).sum())
            pred_k = len(selected)
            random_precision = true_n / float(n)
            precision = hits / float(pred_k) if pred_k else np.nan
            universe_alpha = float(month[TARGET_COL].mean())
            top8_alpha = float(selected[TARGET_COL].mean()) if pred_k else np.nan
            row = dict(common_meta)
            row.update({
                "rebalance_date": pd.Timestamp(date_value), "year": int(pd.Timestamp(date_value).year),
                "variant": variant, "n": n, "selected_count": pred_k,
                "rank_ic": safe_rank_ic(month[TARGET_COL], month[score_col]),
                "auc_true_top20": binary_rank_auc(binary_true, month[score_col].values),
                "precision_at8_true20": precision,
                "random_precision": random_precision,
                "precision_lift": precision / random_precision if random_precision > 0 and not pd.isnull(precision) else np.nan,
                "recall_at8_true20": hits / float(true_n) if true_n else np.nan,
                "map_at8_true20": binary_map_at_k(binary_true, month[score_col].values, PRED_TOP_K),
                "ndcg_at8_true20": binary_ndcg_at_k(binary_true, month[score_col].values, PRED_TOP_K),
                "top8_alpha": top8_alpha,
                "top8_edge": top8_alpha - universe_alpha if pred_k else np.nan,
                "universe_alpha": universe_alpha, "turnover": turnover,
            })
            metric_rows.append(row)
            if store_selected:
                for _, selected_row in selected.iterrows():
                    detail = dict(common_meta)
                    detail.update({
                        "rebalance_date": pd.Timestamp(date_value), "variant": variant,
                        "stock": selected_row[STOCK_COL], "score": selected_row[score_col],
                        "alpha_1m": selected_row[TARGET_COL], "true_rank": selected_row["true_rank"],
                        "board": stock_board(selected_row[STOCK_COL]),
                    })
                    selected_rows.append(detail)
    return metric_rows, selected_rows


## 9. Money2 baseline 与 P2/P3/P4 年度 walk-forward OOS

In [ ]:
def variant_features(train_df, incremental_cols):
    base_features, removed_base = select_v46_features(train_df)
    incremental_features, removed_incremental, coverage_map = select_incremental_features(train_df, incremental_cols)
    return unique_keep_order(base_features + incremental_features), removed_base, removed_incremental, coverage_map


family_gate_map = dict(zip(coverage_df["family"], coverage_df["data_gate"]))
VARIANTS = [{"variant": "money2", "family": "money2_baseline", "incremental": MONEY2_BASELINE_COLS}]
for family, features in family_features.items():
    if bool(family_gate_map.get(family, False)):
        VARIANTS.append({"variant": "money2_plus_" + family.split("_")[0], "family": family, "incremental": unique_keep_order(MONEY2_BASELINE_COLS + features)})
available_families = [v for v in VARIANTS if v["variant"] != "money2"]
if len(available_families) >= 2:
    combined = unique_keep_order(MONEY2_BASELINE_COLS + [c for v in available_families for c in family_features[v["family"]]])
    VARIANTS.append({"variant": "money2_plus_all_available", "family": "all_available", "incremental": combined})

variant_manifest_df = pd.DataFrame([{
    "variant": v["variant"], "family": v["family"], "candidate_feature_count": len(v["incremental"]),
    "candidate_features": ",".join(v["incremental"]),
} for v in VARIANTS])
variant_manifest_df.to_csv(OUT_DIR / "v98_variant_manifest.csv", index=False)
display_df(variant_manifest_df, 20)

monthly_rows = []
model_rows = []
importance_rows = []
selected_rows = []
tasks = [(cutoff, spec) for cutoff in CORE_CUTOFFS for spec in VARIANTS]
for cutoff, spec in progress_iter(tasks, total=len(tasks), desc="V98 core OOS"):
    train_df, test_df, train_start = build_train_test(cutoff, "expanding")
    if train_df[DATE_COL].nunique() < MIN_TRAIN_MONTHS or len(test_df) == 0:
        continue
    features, removed_base, removed_inc, coverage_map = variant_features(train_df, spec["incremental"])
    meta = {
        "cutoff": cutoff, "train_policy": "expanding", "train_start": train_start,
        "family": spec["family"], "removed_base": ",".join(removed_base),
        "removed_incremental": ",".join(removed_inc),
    }
    score_map, fit_meta, fit_importance, _ = train_predict_many_seeds(
        train_df, test_df, features, CORE_SEEDS, FEATURE_FRACTION, meta,
    )
    model_rows.extend(fit_meta)
    importance_rows.extend(fit_importance)
    for seed in CORE_SEEDS:
        panel = test_df[[STOCK_COL, DATE_COL, TARGET_COL]].copy()
        panel["score"] = score_map[int(seed)]
        metrics, selected = evaluate_score_panel(
            panel, {spec["variant"]: "score"},
            {"cutoff": cutoff, "seed": int(seed), "family": spec["family"]},
            store_selected=(int(seed) == 42),
        )
        monthly_rows.extend(metrics)
        selected_rows.extend(selected)
        del panel
    del train_df, test_df, score_map
    gc.collect()

oos_monthly_df = pd.DataFrame(monthly_rows)
model_meta_df = pd.DataFrame(model_rows)
feature_importance_df = pd.DataFrame(importance_rows)
selected_df = pd.DataFrame(selected_rows)
oos_monthly_df.to_csv(OUT_DIR / "v98_oos_monthly.csv", index=False)
model_meta_df.to_csv(OUT_DIR / "v98_model_meta.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v98_feature_importance.csv", index=False)
selected_df.to_csv(OUT_DIR / "v98_selected_top8.csv", index=False)

# Exact-baseline gate: when V96 core output is available, V98 Money2 must reproduce it.
v96_reference_candidates = [
    PANEL_PATH.parent / "v96_core_monthly.csv",
    PROJECT_DIR / "csi800_ml_v96_money_quality_outputs" / "v96_core_monthly.csv",
    PROJECT_DIR / "v96_core_monthly.csv",
]
v96_reference_path = None
for candidate in v96_reference_candidates:
    if candidate.exists() and candidate.is_file():
        v96_reference_path = candidate
        break
baseline_reconciliation_df = pd.DataFrame([{
    "status": "reference_not_found", "reference_path": "", "matched_rows": 0,
    "expected_rows": int((oos_monthly_df["variant"] == "money2").sum()),
    "rank_ic_max_abs_delta": np.nan, "top8_edge_max_abs_delta": np.nan,
    "rank_ic_atol": BASELINE_RANK_IC_ATOL, "top8_edge_atol": BASELINE_TOP8_EDGE_ATOL,
}])
if v96_reference_path is not None:
    reference = pd.read_csv(str(v96_reference_path), low_memory=False)
    reference = reference[reference["variant"] == "money2_all"].copy()
    reference[DATE_COL] = pd.to_datetime(reference[DATE_COL], errors="coerce")
    reference["cutoff"] = reference["cutoff"].astype(str)
    baseline_v98 = oos_monthly_df[oos_monthly_df["variant"] == "money2"].copy()
    baseline_v98["cutoff"] = baseline_v98["cutoff"].astype(str)
    keys = ["cutoff", "seed", DATE_COL]
    matched = baseline_v98.merge(
        reference[keys + ["rank_ic", "top8_edge"]], on=keys, how="inner",
        suffixes=("_v98", "_v96"),
    )
    rank_delta = float((matched["rank_ic_v98"] - matched["rank_ic_v96"]).abs().max()) if len(matched) else np.nan
    edge_delta = float((matched["top8_edge_v98"] - matched["top8_edge_v96"]).abs().max()) if len(matched) else np.nan
    baseline_reconciliation_df = pd.DataFrame([{
        "status": "ok" if (
            len(matched) == len(baseline_v98) and
            rank_delta <= BASELINE_RANK_IC_ATOL and
            edge_delta <= BASELINE_TOP8_EDGE_ATOL
        ) else "fail",
        "reference_path": str(v96_reference_path), "matched_rows": len(matched),
        "expected_rows": len(baseline_v98), "rank_ic_max_abs_delta": rank_delta,
        "top8_edge_max_abs_delta": edge_delta,
        "rank_ic_atol": BASELINE_RANK_IC_ATOL, "top8_edge_atol": BASELINE_TOP8_EDGE_ATOL,
    }])
    if baseline_reconciliation_df["status"].iloc[0] != "ok":
        raise RuntimeError("V98 Money2 baseline does not reproduce V96: %s" % baseline_reconciliation_df.to_dict("records"))
baseline_reconciliation_df.to_csv(QA_DIR / "v98_money2_baseline_reconciliation.csv", index=False)


## 10. 相邻训练截止日稳定性

In [ ]:
def score_variant_for_window(cutoff, spec, eval_start, eval_end):
    train_df, test_df, train_start = build_train_test(cutoff, "expanding", eval_start, eval_end)
    if len(train_df) == 0 or len(test_df) == 0:
        return pd.DataFrame()
    features, _, _, _ = variant_features(train_df, spec["incremental"])
    scores, _, _, _ = train_predict_many_seeds(
        train_df, test_df, features, [UPDATE_SEED], FEATURE_FRACTION,
        {"cutoff": cutoff, "family": spec["family"], "train_policy": "expanding", "train_start": train_start},
    )
    panel = test_df[[STOCK_COL, DATE_COL, TARGET_COL]].copy()
    panel["score"] = scores[int(UPDATE_SEED)]
    del train_df, test_df, scores
    gc.collect()
    return panel


update_rows = []
if RUN_UPDATE_STABILITY:
    tasks = [(pair, spec) for pair in UPDATE_PAIRS for spec in VARIANTS]
    for pair, spec in progress_iter(tasks, total=len(tasks), desc="adjacent cutoff stability"):
        old_panel = score_variant_for_window(pair["old_cutoff"], spec, pair["eval_start"], pair["eval_end"])
        new_panel = score_variant_for_window(pair["new_cutoff"], spec, pair["eval_start"], pair["eval_end"])
        if len(old_panel) == 0 or len(new_panel) == 0:
            continue
        merged = old_panel.merge(new_panel[[STOCK_COL, DATE_COL, "score"]], on=[STOCK_COL, DATE_COL], suffixes=("_old", "_new"))
        for date_value, month in merged.groupby(DATE_COL):
            old_indices = select_top8_board_capped(month, "score_old")
            new_indices = select_top8_board_capped(month, "score_new")
            old_set = set(month.loc[old_indices, STOCK_COL])
            new_set = set(month.loc[new_indices, STOCK_COL])
            update_rows.append({
                "pair_id": pair["pair_id"], "variant": spec["variant"], "family": spec["family"],
                DATE_COL: pd.Timestamp(date_value),
                "score_rank_corr": safe_rank_ic(month["score_old"], month["score_new"]),
                "top8_overlap": len(old_set.intersection(new_set)) / float(PRED_TOP_K),
            })
        del old_panel, new_panel, merged
        gc.collect()
update_stability_df = pd.DataFrame(update_rows)
update_stability_df.to_csv(OUT_DIR / "v98_adjacent_cutoff_stability_monthly.csv", index=False)


## 11. 汇总、配对 block bootstrap 与决策表

In [ ]:
METRIC_COLS = [
    "rank_ic", "auc_true_top20", "precision_at8_true20", "precision_lift",
    "recall_at8_true20", "map_at8_true20", "ndcg_at8_true20",
    "top8_alpha", "top8_edge", "universe_alpha", "turnover",
]


def summarize_metrics(frame, group_cols):
    rows = []
    if frame is None or not len(frame):
        return pd.DataFrame()
    grouped = list(frame.groupby(group_cols))
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize", leave=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row["months"] = part[DATE_COL].nunique() if DATE_COL in part.columns else len(part)
        for col in METRIC_COLS:
            if col in part.columns:
                values = pd.to_numeric(part[col], errors="coerce")
                row[col + "_mean"] = float(values.mean())
                row[col + "_std"] = float(values.std())
        rows.append(row)
    return pd.DataFrame(rows)


def block_bootstrap_probability(values, samples, block_months, seed):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(int(seed))
    means = []
    starts = np.arange(n)
    for _ in range(int(samples)):
        picked = []
        while len(picked) < n:
            start = int(rng.choice(starts))
            picked.extend([values[(start + j) % n] for j in range(int(block_months))])
        means.append(float(np.mean(picked[:n])))
    means = np.asarray(means)
    return float((means > 0).mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


oos_summary_df = summarize_metrics(oos_monthly_df, ["variant"])
oos_yearly_df = summarize_metrics(oos_monthly_df, ["variant", "year"])
oos_seed_df = summarize_metrics(oos_monthly_df, ["variant", "seed"])

pair_keys = ["cutoff", "seed", DATE_COL]
baseline = oos_monthly_df[oos_monthly_df["variant"] == "money2"][pair_keys + ["rank_ic", "top8_edge"]]
increment_rows = []
decision_rows = []
update_summary_rows = []
if len(update_stability_df):
    for variant, part in update_stability_df.groupby("variant"):
        update_summary_rows.append({
            "variant": variant, "months": len(part),
            "score_rank_corr_mean": float(part["score_rank_corr"].mean()),
            "top8_overlap_mean": float(part["top8_overlap"].mean()),
        })
update_summary_df = pd.DataFrame(update_summary_rows)

for spec in VARIANTS:
    variant = spec["variant"]
    if variant == "money2":
        continue
    candidate = oos_monthly_df[oos_monthly_df["variant"] == variant][pair_keys + ["rank_ic", "top8_edge"]]
    paired = candidate.merge(baseline, on=pair_keys, suffixes=("", "_baseline"))
    paired["delta_rank_ic"] = paired["rank_ic"] - paired["rank_ic_baseline"]
    paired["delta_top8_edge"] = paired["top8_edge"] - paired["top8_edge_baseline"]
    paired["variant"] = variant
    increment_rows.append(paired)
    year_delta = paired.groupby(paired[DATE_COL].dt.year)["delta_top8_edge"].mean()
    seed_delta = paired.groupby("seed")["delta_top8_edge"].mean()
    monthly_delta = paired.groupby(DATE_COL)["delta_top8_edge"].mean().sort_index()
    probability, ci_low, ci_high = block_bootstrap_probability(
        monthly_delta.values, BOOTSTRAP_SAMPLES, BOOTSTRAP_BLOCK_MONTHS, BOOTSTRAP_SEED,
    )
    update_part = update_summary_df[update_summary_df["variant"] == variant] if len(update_summary_df) else pd.DataFrame()
    update_corr = float(update_part["score_rank_corr_mean"].iloc[0]) if len(update_part) else np.nan
    update_overlap = float(update_part["top8_overlap_mean"].iloc[0]) if len(update_part) else np.nan
    checks = {
        "delta_edge": float(paired["delta_top8_edge"].mean()) >= ACCEPT_MIN_DELTA_TOP8_EDGE,
        "delta_rank_ic": float(paired["delta_rank_ic"].mean()) >= ACCEPT_MIN_DELTA_RANK_IC,
        "positive_years": int((year_delta > 0).sum()) >= ACCEPT_MIN_POSITIVE_YEARS,
        "positive_seeds": int((seed_delta > 0).sum()) >= ACCEPT_MIN_POSITIVE_SEEDS,
        "bootstrap": probability >= ACCEPT_MIN_BOOTSTRAP_PROBABILITY,
        "update_corr": (not RUN_UPDATE_STABILITY) or (not pd.isnull(update_corr) and update_corr >= ACCEPT_MIN_UPDATE_SCORE_CORR),
        "update_overlap": (not RUN_UPDATE_STABILITY) or (not pd.isnull(update_overlap) and update_overlap >= ACCEPT_MIN_UPDATE_TOP8_OVERLAP),
    }
    decision_rows.append({
        "variant": variant, "family": spec["family"],
        "delta_top8_edge_mean": float(paired["delta_top8_edge"].mean()),
        "delta_rank_ic_mean": float(paired["delta_rank_ic"].mean()),
        "positive_years": int((year_delta > 0).sum()), "positive_seeds": int((seed_delta > 0).sum()),
        "bootstrap_probability_positive": probability, "bootstrap_ci_low": ci_low, "bootstrap_ci_high": ci_high,
        "update_score_corr": update_corr, "update_top8_overlap": update_overlap,
        "passed_checks": int(_bi.sum([bool(x) for x in checks.values()])),
        "total_checks": len(checks), "accept_for_candidate": bool(_bi.all(list(checks.values()))),
        "failed_checks": ",".join([name for name, passed in checks.items() if not passed]),
    })

increment_df = pd.concat(increment_rows, ignore_index=True, sort=False) if increment_rows else pd.DataFrame()
decision_df = pd.DataFrame(decision_rows)

oos_summary_df.to_csv(OUT_DIR / "v98_oos_summary.csv", index=False)
oos_yearly_df.to_csv(OUT_DIR / "v98_oos_yearly.csv", index=False)
oos_seed_df.to_csv(OUT_DIR / "v98_oos_seed.csv", index=False)
increment_df.to_csv(OUT_DIR / "v98_increment_vs_money2.csv", index=False)
update_summary_df.to_csv(OUT_DIR / "v98_adjacent_cutoff_stability_summary.csv", index=False)
decision_df.to_csv(OUT_DIR / "v98_decision_table.csv", index=False)
display_df(oos_summary_df, 20)
display_df(decision_df, 20)


## 12. 可视化与输出完整性门禁

In [ ]:
if plt is not None and len(oos_monthly_df):
    plot_df = oos_monthly_df.groupby([DATE_COL, "variant"])[["rank_ic", "top8_edge", "precision_lift", "map_at8_true20"]].mean().reset_index()
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for variant, part in plot_df.groupby("variant"):
        part = part.sort_values(DATE_COL)
        axes[0, 0].plot(part[DATE_COL], part["top8_edge"].rolling(12, min_periods=6).mean(), label=variant)
        axes[0, 1].plot(part[DATE_COL], part["rank_ic"].rolling(12, min_periods=6).mean(), label=variant)
        axes[1, 0].plot(part[DATE_COL], part["precision_lift"].rolling(12, min_periods=6).mean(), label=variant)
        axes[1, 1].plot(part[DATE_COL], part["map_at8_true20"].rolling(12, min_periods=6).mean(), label=variant)
    titles = ["Rolling 12m Top8 edge", "Rolling 12m RankIC", "Rolling 12m Precision lift", "Rolling 12m MAP@8"]
    for ax, title in zip(axes.ravel(), titles):
        ax.set_title(title)
        ax.axhline(0, color="black", linewidth=0.7)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(str(FIG_DIR / "v98_rolling_oos_metrics.png"), dpi=150)
    plt.show()

output_files = [
    "v98_feature_contract.csv", "v98_analyst_data_gate.csv", "v98_source_preflight.csv",
    "v98_family_source_coverage.csv", "v98_external_information_enriched_panel.csv",
    "v98_variant_manifest.csv", "v98_oos_monthly.csv", "v98_oos_summary.csv",
    "v98_oos_yearly.csv", "v98_oos_seed.csv", "v98_increment_vs_money2.csv",
    "v98_model_meta.csv", "v98_feature_importance.csv", "v98_selected_top8.csv",
    "v98_adjacent_cutoff_stability_monthly.csv", "v98_adjacent_cutoff_stability_summary.csv",
    "v98_decision_table.csv",
]
output_manifest_df = pd.DataFrame([{"file": name, "exists": (OUT_DIR / name).exists()} for name in output_files])
output_manifest_df.to_csv(OUT_DIR / "v98_output_manifest.csv", index=False)
missing_outputs = output_manifest_df.loc[~output_manifest_df["exists"], "file"].tolist()
if missing_outputs:
    raise RuntimeError("V98 missing outputs: %s" % missing_outputs)
if int(pit_audit_df["violations"].sum()) > 0:
    raise RuntimeError("V98 point-in-time audit failed; do not interpret model results")
print("V98 COMPLETE")
print("P1 is intentionally not trained unless a verified row-level analyst source is configured.")
print("Send the whole csi800_ml_v98_external_information_outputs folder.")
